# E-Commerce Purchase Intention Prediction

## Objective
Predict whether an e-commerce visitor will complete a purchase based on session-level behavior available before the transaction.

The target is `Revenue`, and we explicitly exclude `PageValues` because it is a post-conversion signal that would leak the outcome information into the model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/online_shoppers_intention.csv')
df.head()


## 1. Initial dataset inspection
We inspect the schema, row count, target, missing values, and duplicate rows before modeling.


In [ ]:
print('shape:', df.shape)
print('dtypes:\n', df.dtypes)
print('missing_values:', df.isna().sum().sum())
print('duplicate_rows:', df.duplicated().sum())
print('target_distribution:\n', df['Revenue'].value_counts())


## 2. Data quality assessment
The dataset is complete and not duplicated, but the target is imbalanced with only about 15% purchase sessions.


In [ ]:
target_counts = df['Revenue'].value_counts(normalize=True)
print(target_counts)
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x='Revenue', palette='Set2')
plt.title('Revenue distribution')
plt.show()


## 3. Exploratory Data Analysis
We review the behavioral predictors that most likely matter for conversion behavior.


In [ ]:
for col in ['BounceRates', 'ExitRates', 'ProductRelated_Duration', 'SpecialDay']:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], bins=25, kde=True)
    plt.title(col)
    plt.show()

plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='VisitorType', hue='Revenue', palette='Set2')
plt.title('Visitor type vs Revenue')
plt.show()


## 4. Feature leakage investigation
`PageValues` is the most obvious leakage risk: it represents the transaction value for completed purchases and is therefore not available before the purchase decision.


In [ ]:
print('PageValues mean by target:\n', df.groupby('Revenue')['PageValues'].mean())
print('Share of non-zero PageValues:', (df['PageValues'] > 0).mean())
print('Feature excluded from final model: PageValues')


## 5. Preprocessing and train/test split
The final feature set excludes leakage-prone columns and encodes the remaining categorical variables.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

feature_cols = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates',
    'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend'
]
X = df[feature_cols]
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
])


## 6. Baseline model
We start with logistic regression as a simple benchmark for interpretability and performance.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear'))
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_proba = baseline.predict_proba(X_test)[:, 1]
print({
    'accuracy': accuracy_score(y_test, baseline_pred),
    'precision': precision_score(y_test, baseline_pred, zero_division=0),
    'recall': recall_score(y_test, baseline_pred, zero_division=0),
    'f1': f1_score(y_test, baseline_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, baseline_proba)
})


## 7. Multiple model experiments
We compare several model families and keep the test set hidden until the final evaluation.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
import pandas as pd

models = {
    'logreg': LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear'),
    'rf': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced_subsample'),
    'hgb': HistGradientBoostingClassifier(random_state=42, learning_rate=0.05, max_depth=6, max_iter=400),
    'xgb': XGBClassifier(
        objective='binary:logistic', n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.8, eval_metric='logloss', random_state=42
    )
}

results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    results.append({
        'name': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba)
    })

pd.DataFrame(results).sort_values('roc_auc', ascending=False)


## 8. Hyperparameter tuning
We tune the best candidate using validation folds and `roc_auc` as the selection metric.


In [ ]:
from sklearn.model_selection import GridSearchCV

final_model = HistGradientBoostingClassifier(random_state=42)
final_pipeline = Pipeline([('preprocessor', preprocessor), ('model', final_model)])
param_grid = {
    'model__max_depth': [3, 4, 5],
    'model__learning_rate': [0.03, 0.05, 0.08],
    'model__max_iter': [200, 400],
    'model__min_samples_leaf': [10, 20],
    'model__l2_regularization': [0.0, 0.1]
}
search = GridSearchCV(final_pipeline, param_grid=param_grid, scoring='roc_auc', cv=3)
search.fit(X_train, y_train)
search.best_params_


## 9. Final model selection and evaluation
The selected pipeline is evaluated on the held-out test set only once, preserving a clean estimate of generalization performance.


In [ ]:
best_model = search.best_estimator_
final_pred = best_model.predict(X_test)
final_prob = best_model.predict_proba(X_test)[:, 1]

final_metrics = {
    'accuracy': accuracy_score(y_test, final_pred),
    'precision': precision_score(y_test, final_pred, zero_division=0),
    'recall': recall_score(y_test, final_pred, zero_division=0),
    'f1': f1_score(y_test, final_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, final_prob)
}
print(final_metrics)


## 10. Conclusions and observations
The model succeeds at identifying likely non-conversions and provides a useful probability estimate for purchase intent, while remaining honest about the class imbalance and the business-use case.

This approach is suitable for business-facing analytics and decision support, but not as a causal proof of conversion behavior.
